# Run DynaGuide in the CALVIN env

## Prerequisites

1. Install the env once with `setup_dynaguide.bat`, see [tutorial_for_pclab.md](../tutorial_for_pclab.md)
1. Download the DynaGuide model from this [link](https://drive.google.com/file/d/1DeOnoDacXjBHgy1DGoRJpYIR8SyDy5fJ/view?usp=drive_link)
1. Download the base policy from this [link](https://drive.google.com/file/d/1lcI_PBgFIYDsoK4T4qO7SGJJgI2lw0Kd/view?usp=drive_link)
1. Download the guidance conditions for `switch_on` from this [link](https://drive.google.com/file/d/1wtEGnG87Y-imqD2MygcqbJwp_RGi7YNB/view?usp=drive_link)
1. Put these three downloaded files in the folder _DynaGuide-main_, keeping their names. `calvin_exp_configs_examples/switch_on.json` already points to the guidance condition there, so nothing needs editing

## Downloading datasets

See this link for further details: https://github.com/mees/calvin/tree/main/dataset
### Downloading task_D_D
    wget http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip
    unzip task_D_D.zip

### Downloading task_ABC_D
    wget http://calvin.cs.uni-freiburg.de/dataset/task_ABC_D.zip
    unzip task_ABC_D.zip

### Downloading task_ABCD_D
    wget http://calvin.cs.uni-freiburg.de/dataset/task_ABCD_D.zip
    unzip task_ABCD_D.zip

### Downloading the debug dataset
    wget http://calvin.cs.uni-freiburg.de/dataset/calvin_debug_dataset.zip
    unzip calvin_debug_dataset.zip

## Running DynaGuide rollouts

Now, run this command in the Windows Anaconda Prompt. The `^` at the end of a line continues the command on the next line, like `\` on Linux. Nothing may follow it, not even a space, and the last line has none. While pasting, the prompt shows `More?` for each continued line, which is normal. You can also paste the code into a .bat file and run that.
```bat
conda activate dynaguide
cd DynaGuide-main
python dynaguide_scripts\run_dynaguide.py ^
    --output_folder results\switch_on_test ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\switch_on.json ^
    --horizon 400 ^
    --n_rollouts 3 ^
    --video_skip 2 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --save_frames
```

The paths are relative, so it has to run from inside `DynaGuide-main`. A PyBullet window opens during the rollouts, leave it open.

Everything from the run lands in `--output_folder`, which is created if it doesn't exist: `<date>_<time>_video.mp4`, `<date>_<time>_rollout.hdf5`, `<date>_<time>_log.json`, `args.json` and the frames from `--save_frames`. Use a new folder for each run. If the folder already holds results, the run stops right away so nothing gets overwritten.

**Faster.** This has the same physics and the same images for the policy, it only changes what you see. The video is still saved:
```bat
python dynaguide_scripts\run_dynaguide.py ^
    --output_folder results\switch_on_fast ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\switch_on.json ^
    --horizon 400 ^
    --n_rollouts 3 ^
    --video_skip 4 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --no_live_render ^
    --no_collage
```

| Option | What it does | What you lose |
|---|---|---|
| `--no_live_render` | No PyBullet window. Without EGL (Windows, Mac) the camera images come from the CPU renderer with or without the window, so they stay the same pixel for pixel | You don't see the robot live, watch the video afterwards. The console prints the progress |
| `--no_collage` | Skips the dynamics model reconstructions | The tiles above the video and in `--save_frames` |
| `--video_skip 4` | Saves every 4th step instead of every 2nd | The video is choppier and shorter |

`Time_Policy`, `Time_Env`, `Time_Collage` and `Time_Video` in the log show the average seconds per rollout for each part, and `time` the total. `Time_Policy` is the guidance itself and can't be reduced without changing DynaGuide.

To get more rollouts per hour, you can also start a second run in another Anaconda Prompt with its own `--output_folder` and `--seed`. The GTX 1650 has 4 GB of memory, so check with `nvidia-smi` that both fit.

To record many guided rollouts unattended, for new guidance conditions, see section 5 of `Create guidance conditions.ipynb`.

## Experiment configs

An experiment config is the json file you pass with `--exp_setup_config`. It says **what counts as success** (`target_behaviors`), **how each rollout starts** (`env_setup`, `reset_poses`) and **where to steer** (`pos_examples`). The examples are in `calvin_exp_configs_examples`, copy one and change it.

The config looks like this, all fields are explained below:

```json
{
    "target_behaviors" : ["drawer_open"],
    "env_setup" : {"drawer" : "closed", "switch" : "off"},
    "use_neg" : false,
    "pos_examples" : "datasets/guidance_conditions/debug_all/drawer_open.hdf5",
    "reset_poses" : "calvin_exp_configs_examples/reset_poses/initial_calvin_robot_states_right_side_midpoint.json"
}
```

Each rollout of this config starts with the drawer closed and the switch off, everything else random. The robot is steered towards the images in `pos_examples`, and a rollout is a success once the drawer is open.

| Field | Value | If you leave it out |
|---|---|---|
| `target_behaviors` | List of behaviors, see below | No goal: every rollout runs to `--horizon`, only the behavior counts are reported |
| `avoid_behaviors` | List of behaviors, see below | Nothing counts as a failure |
| `env_setup` | The start of the scene, see below. `{}` for all random | **Error**, write at least `{}` |
| `use_neg` | `true` or `false`: also steer away from `neg_examples` | **Error** |
| `pos_examples` | Guidance conditions to steer towards: a path to a condition file (hdf5), a list of paths, or patterns with `*`, see below. `null` for none. Relative to `DynaGuide-main` | **Error**, write `null` for no conditions |
| `neg_examples` | Guidance conditions to steer away from, written like `pos_examples` | Only needed with `"use_neg" : true` |
| `reset_poses` | Path to a json file of robot start poses, one is picked at random per rollout. The files are in `calvin_exp_configs_examples/reset_poses` | **Error**. With `null`, the robot always starts in CALVIN's default pose |
| `loc_target` | `[x, y, z]`, a position to steer the gripper to | Only needed with `--direct_position_guidance` |

**Several condition files.** `pos_examples` and `neg_examples` can take a list, and each entry can be a pattern with `*`. All matching files are used together, as if they were one file. This way you combine conditions without making a merged copy:

```json
"pos_examples" : [
    "datasets/guidance_conditions/recombined_debug/drawer_open/drawer_open__door_*__switch_off__green_light_off.hdf5",
    "datasets/guidance_conditions/recombined_debug_no_blocks/drawer_open/drawer_open__door_*__switch_off__green_light_off.hdf5"
]
```

Here `door_*` matches `door_left` and `door_right`, so this uses 4 files with 16 conditions. At the start of a run, `run_dynaguide.py` prints which files it uses, and a path or pattern without a matching file stops the run with an error. Only `run_dynaguide.py` reads lists, `run_sampling_baseline.py` still takes a single file.


### 1. `target_behaviors` and `avoid_behaviors`: when a rollout succeeds

Both are lists of behavior names. These are the names you can use:

| Part of the scene | Behaviors |
|---|---|
| Drawer | `drawer_open`, `drawer_close` |
| Sliding door | `door_left`, `door_right` |
| Switch (lever, yellow light) | `switch_on`, `switch_off` |
| Button (green light) | `button_on`, `button_off` |
| Blocks | `red_displace`, `blue_displace`, `pink_displace` |
| Paper's rules | the same names with `sanity_` in front, and `sanity_any`, see "Replicating the paper" |

Examples:

| You write | A rollout is a success when | See example config |
|---|---|---|
| `"target_behaviors" : ["drawer_open"]` | the drawer is open | `drawer_open.json` |
| `"target_behaviors" : ["button_on", "switch_on"]` | **any one** of them happened | `multiple_and_negative_objectives/button_on_switch_on.json` |
| `"avoid_behaviors" : ["door_right"]` | the door didn't move right until `--horizon`. Doing nothing counts too, so check `no_behavior` in the results | `door_dont_right.json` |
| both (target & avoid) | a target happened and no avoid behavior happened | |
| neither | there is no success, only the counts | `control.json` |

A rollout ends as soon as a target or something to avoid happens. Other behaviors on the way don't end it.

The goal only decides success and when to stop. **It doesn't steer the robot**, `pos_examples` does that. So the conditions should show the same behavior as the goal.

### 2. `env_setup`: how the scene starts

Every part you write in `env_setup` starts the same in every rollout. Every part you leave out is random in each rollout.

| Parameter | Values | Example | If you leave it out |
|---|---|---|---|
| `drawer` | `"open"` (0.16 m) / `"closed"` (0), or how far it is pulled out in meters, 0 to 0.22 | `"drawer" : "closed"` | Open or closed, 50 / 50 |
| `sliding_door` | `"left"` (0.27 m) / `"right"` (0), or a position in meters, 0 to 0.28. The door always covers one of the two cabinet compartments | `"sliding_door" : "left"` | Left or right, 50 / 50 |
| `switch` | `"on"` (0.09 m) / `"off"` (0), or a position in meters, 0 to 0.09. The yellow light follows, it is on above about 0.045 | `"switch" : "off"` | On or off, 50 / 50 |
| `green_light` | `"on"` / `"off"`, or `1` / `0`. The light the button toggles | `"green_light" : "off"` | On or off, 50 / 50 |
| `blocks` | Where each block goes, see below | `"blocks" : {"red" : "drawer"}` | Each block lies at a random place and angle on the table or in one of the two cabinet compartments, never in the drawer |
| `cubes_table_only` | `true`, found in the paper's configs | | Nothing changes, it has no effect |

**The goal's part has to start where the goal can happen.** For `drawer_open` the drawer has to start closed, for `switch_on` the switch off, for `button_on` the green light off. The exact limits are in the "Start" column of "Behaviors and thresholds". If you leave the goal's part out, about half the rollouts start with the goal already done and can never succeed, so always set it.

```json
"env_setup" : {"drawer" : "closed", "sliding_door" : "left", "switch" : "off", "green_light" : "on"}
```

#### 2.1 `blocks`

Each of `red`, `blue` and `pink` gets one of three forms. Blocks you leave out are random as above:

```json
"env_setup" : {
    "drawer" : "open",
    "blocks" : {
        "red" : "table_center",
        "blue" : {"place" : "drawer", "yaw" : 0.5},
        "pink" : {"pos" : [0.1, -0.1, 0.46], "yaw" : 0.0}
    }
}
```

| Form | Example | Meaning |
|---|---|---|
| A place | `"red" : "table_center"` | One of the places below |
| A place with options | `"blue" : {"place" : "drawer", "yaw" : 0.5, "offset" : [0.02, 0.0]}` | `yaw`: the angle in radians. `offset`: moves the block by `[dx, dy]` meters. Both are optional |
| Coordinates | `"pink" : {"pos" : [0.1, -0.1, 0.46], "yaw" : 0.0}` | The center of the block. `yaw` optional, 0 if left out |

| Where | Random position and angle inside | Fixed position, angle 0 |
|---|---|---|
| Table | `table` | `table_left`, `table_center`, `table_right` |
| Drawer | `drawer` | `drawer_left`, `drawer_center`, `drawer_right` |
| Cabinet, left compartment | `cabinet_left` | `cabinet_left_center` |
| Cabinet, right compartment | `cabinet_right` | `cabinet_right_center` |
| Not in the scene | | `hidden`, parked outside the camera view |

- Drawer places move with the drawer, whatever its state. A block in a closed drawer can't be seen.
- Coordinates don't move with the drawer. For your own position in the drawer, use a drawer place with an `offset`.
- A block in the compartment behind the sliding door can't be seen either.
- `"blocks" : {"red" : "hidden", "blue" : "hidden", "pink" : "hidden"}` gives a table without blocks.

### 3. Checking a config

`run_dynaguide.py` checks the config before it loads anything. It stops with an error for an unknown key, behavior or place, for blocks that would overlap, and for a goal that can't happen from the start, like `drawer_open` with `"drawer" : "open"`.

To see the start of the rollouts without running a policy, render a set of resets. The images go into a new folder inside `results/reset_previews`, or inside `--output_folder` if you pass one. `resets_start.png` and `resets_settled.png` show the resets from the camera of the policy. Resets where a block falls, shifts, intersects another block or touches the robot get a red border and are listed in `report.json`:

```bash
python dynaguide_scripts/preview_resets.py --config calvin_exp_configs_examples/reset_layouts/blocks_in_all_places.json
```

## Evaluating rollout success automatically

CALVIN has no success signal of its own, so `core/calvin_behaviors.py` detects what the robot achieved from the states and compares this to the goal in the experiment config. You don't need a separate analysis step or config. After the run you can watch the video and read the result in the terminal and in `<date>_<time>_log.json` in the output folder:

```json
"Target_Behaviors": ["switch_on"],
"Avoid_Behaviors": [],
"Success_Rate": 0.6667,
"Num_Success": 2,
"Num_Episode": 3,
"Behavior_Counts": {"switch_on": 2, "door_left": 1},
"Episode_Behaviors": [["switch_on"], ["door_left"], ["switch_on"]]
```

- `Success_Rate`, `Num_Success`: the rollouts that reached the goal. Both are `null` if the config has no goal, like `control.json`
- `Behavior_Counts`: how often each behavior happened. This also tells you what the robot did instead when it failed
- `Episode_Behaviors`: the behavior of each rollout, in order, so you can look up a specific rollout in the video

### Scoring a file you already have

This needs no rollouts. Pass the output folder of the run, or a single hdf5 file, which needs `--dataset_obs` if it comes from another script. Files from `run_trained_agent.py`, `run_sampling_baseline.py` and older `run_dynaguide.py` runs stop at the first small change, usually before the goal is reached, so they mostly score `no_behavior`:

```bat
python dynaguide_scripts\evaluate_behaviors.py ^
    --dataset results\switch_on_test ^
    --config calvin_exp_configs_examples\switch_on.json ^
    --output results\switch_on_test\behaviors.json
```

### Running other tasks

Pass a different config with `--exp_setup_config`. All configs in `calvin_exp_configs_examples` already have their goal, so only `pos_examples` / `neg_examples` need to point to your guidance conditions. Only the `switch_on` conditions can be downloaded, make the others with `Create guidance conditions.ipynb`.

### When a rollout ends

The config decides this, see "Experiment configs" below. `run_dynaguide.py` only asks `GoalTracker` in `core/calvin_behaviors.py` whether the outcome is decided. Without a target or something to avoid, a rollout ends after `--horizon` steps. This is the only way to end for `control.json`, and for avoid-only configs it means success.

If the robot opens the drawer on the way to the switch, it continues and can still reach the switch. `Episode_Behaviors` lists everything it achieved, in order. So set `--horizon` high enough for the task, and expect failed rollouts to take the full time.

These results aren't comparable to the paper's numbers. The paper stops at the first small change and measures which object was touched first, while here the robot keeps trying until it reaches the goal. To compare with the paper, use the `sanity_` behaviors, see "Replicating the paper" below. For a baseline, run the same config with `--scale 0 --ss 1`.

### Behaviors and thresholds

A behavior counts when the joint starts in the start range and reaches the end range, measured from the first state of the rollout. The `GoalTracker` looks at the state **after** every step, so a goal counts in the step it happens, including the very last step of the horizon.

**The task has to be finished, not only started.** The start ranges are the authors' thresholds from `data_processing_calvin/calvin_to_labeled_hdf5.py` (`ARTICULATED` in `core/calvin_behaviors.py`). Their end thresholds only show that the motion got going: a drawer counted as open at 0.10, about half way, and as closed with 4 cm still open. So the end ranges are now `COMPLETED`, close to where each joint comes to rest in the debug set, with a margin so a joint that stops a little short still counts:

| Behavior | Start | End (finished) | Resting position in the debug set | Old end (authors) |
|---|---|---|---|---|
| `door_left` / `door_right` | below 0.05 / above 0.25 | above **0.26** / below **0.035** | left 0.272 to 0.28 / right 0 to 0.025 | 0.25 / 0.05 |
| `drawer_open` / `drawer_close` | below 0.05 / above 0.10 | above **0.15** / below **0.015** | open 0.178 to 0.217 / closed 0 | 0.10 / 0.05 |
| `switch_on` / `switch_off` | below 0.01 / above 0.07 | above **0.08** / below **0.008** | on 0.088 / off 0 | 0.07 / 0.01 |
| `button_on` / `button_off` | green light below 0.2 / above 0.8 | above 0.8 / below 0.2 | 1 / 0 | unchanged, the light is either 0 or 1 |

The guidance conditions show these finished states: their last frames are at the resting positions (drawer 0.18 to 0.22 or 0 to 0.007, door 0.27 to 0.28 or 0 to 0.025, switch 0.088 to 0.091). The conditions are still cut and named with the authors' thresholds (`ARTICULATED`), because that is what decides where a segment of the dataset ends.

Results from before this change aren't comparable: those rollouts stopped at the old end, so scored again with `evaluate_behaviors.py` they mostly give `no_behavior`.
| `red_displace`, `blue_displace`, `pink_displace` | anywhere | moved more than 0.01 with the gripper within 0.1 |
| `no_behavior` | | none of these happened |

The block condition is the weakest one, 0.01 is only a small push. The block labels mean the block was moved, not necessarily lifted, even though the configs are called `red_lift` etc.

To change where a goal counts as finished, change its number in `COMPLETED` (or `BLOCK_MOVED` for the blocks) in `core/calvin_behaviors.py`. Evaluation and termination both follow, because they use the same code. Leave `ARTICULATED` as it is unless you also want the guidance conditions cut differently. To add a new behavior, add its name to `BEHAVIORS` and its check to `GoalTracker.detect`. Then it can be used in any config.

The cell below shows the results of a run. Set the folder and run it from a kernel started in `DynaGuide-main`.

In [ ]:
import glob
import json

output_folder = "results/switch_on_test"
with open(glob.glob(f"{output_folder}/*_log.json")[0], "r") as f:
    results = json.load(f)

print(f"Goal: {results['Target_Behaviors']}, avoid: {results['Avoid_Behaviors']}")
print(f"Success: {results['Num_Success']} / {results['Num_Episode']}")
print(f"Behaviors: {results['Behavior_Counts']}")
for i, behaviors in enumerate(results["Episode_Behaviors"]):
    print(f"Rollout {i}: {behaviors}")

## Replicating the paper

## Running several experiments unattended

[dynaguide_scripts/run_experiments.py](dynaguide_scripts/run_experiments.py) runs a list of experiments one after the other, each with one or more sets of guidance conditions, and the base policy without guidance to compare. It works like the harvest in `Create guidance conditions.ipynb` and shares its code ([dynaguide_scripts/rollout_tasks.py](dynaguide_scripts/rollout_tasks.py)): every task is its own `run_dynaguide.py` process, a status line comes every 2 minutes, a failed task doesn't stop the others, and the same command again continues where it stopped.

The example plan comes in five parts, to run on several machines at the same time: `plan_first_articulated_1.json` to `plan_first_articulated_5.json` in [calvin_exp_configs_examples/experiments](calvin_exp_configs_examples/experiments), next to the six configs they use. The parts in the "Start" column are fixed, everything else is random:

| Experiment | Part | Config | Start | Condition sets |
|---|---|---|---|---|
| `drawer_open` | 1 | `experiments/drawer_open.json` | drawer closed, switch off, green light off | `blocks`, `no_blocks`, `both` |
| `switch_on` | 1 | `experiments/switch_on.json` | switch off, drawer closed, green light off | `blocks`, `no_blocks`, `both`, `paper` (the downloaded 20) |
| `drawer_close` | 3 | `experiments/drawer_close.json` | drawer open, switch off, green light off | `blocks`, `no_blocks`, `both` |
| `button_on` | 2 | `experiments/button_on.json` | green light off, drawer closed, switch off | `blocks`, `no_blocks`, `both`, `debug_all` (the authors' cut, 3) |
| `door_left` | 4 | `experiments/door_left.json` | door right, switch off, green light off | `blocks`, `no_blocks`, `both`, `debug_all` |
| `door_right` | 5 | `experiments/door_right.json` | door left, switch off, green light off | `blocks`, `no_blocks`, `both`, `debug_all` |

The `blocks` / `no_blocks` sets are the recombined conditions whose scene matches the fixed start: both door positions for the drawer, switch and button, both drawer positions for the doors. `both` uses the two together, `debug_all` is the unchanged file of the debug set. With `"base_policy" : true` every experiment also runs once without guidance (`--scale 0 --ss 1`). That makes 28 tasks with 10 rollouts each: part 1 has 9, part 2 has 5, part 3 has 4, parts 4 and 5 have 5 each. Every experiment has its own `"seed"` (drawer_open 0, drawer_close 1, switch_on 2, button_on 3, door_left 4, door_right 5), so it doesn't matter which machine runs which part. The doors use the paper's settings and reset poses: `door_left` scale 1.5 and alpha 30 from the right side, `door_right` scale 2 and alpha 15 from the middle.

On the first machine:

```bat
python dynaguide_scripts\run_experiments.py --plan calvin_exp_configs_examples\experiments\plan_first_articulated_1.json --output_folder results\experiments_first_1 --dry_run

python dynaguide_scripts\run_experiments.py --plan calvin_exp_configs_examples\experiments\plan_first_articulated_1.json --output_folder results\experiments_first_1 ^
    --agent base_policy_300.pth --guidance dynamics_model_5900.pth
```

On the other machines the same with `plan_first_articulated_2.json` to `_5.json` and `results\experiments_first_2` to `_5`. On one machine, run the parts one after the other. Every machine needs the condition folders of `Create guidance conditions.ipynb`, the fastest is `dynaguide_scripts\make_guidance_conditions.bat` on each.

`--dry_run` lists the tasks with their settings and checks that every config and condition file exists. `--experiments drawer_open switch_on` runs only these.

**The plan:**

| Field | Meaning |
|---|---|
| `n_rollouts`, `horizon` | Per task |
| `scale`, `ss`, `alpha` | Guidance settings, each experiment can set its own. The example uses the paper's per task |
| `base_policy` | `true` adds a run without guidance per experiment |
| `keep_images` | `true` keeps the camera images in the rollout files, to make conditions from them later. Makes the files much bigger |
| `experiments` | `name : {"config" : path, "conditions" : {set name : conditions}, "scale" : ..., "alpha" : ..., "seed" : ...}`. `seed` is optional, by default the experiment's position in the plan. Each set replaces `pos_examples` of the config and is written like it: a path, a list or patterns. Without `conditions`, the config's own `pos_examples` is used, as the set `guided` |

All tasks of one experiment get the same seed, so its condition sets and the base policy start from the same scenes and robot poses, as long as the policy itself doesn't draw from Python's `random`.

**The output folder:**

```
results/experiments_first/
  plan.json                       the plan of this run
  summary.json                    every task with its status and successes, updated after each task
  drawer_open/
    blocks/                       a normal run_dynaguide.py output folder plus config.json, console.txt and DONE
    no_blocks/
    both/
    base_policy/
  ...
  _incomplete/                    tasks that crashed or were interrupted, moved here before they run again
```

At the end it prints a table of all tasks with their successes, which is also in `summary.json`.

The `sanity_` behaviors replicate the paper's numbers instead of checking that the task is finished. Using one switches the whole rollout to the paper's rules:

| | Normal behaviors | `sanity_` behaviors |
|---|---|---|
| Rollout ends | At the goal, or at `--horizon` | At the first small change of **anything**, or at `--horizon` |
| Success | The end state from the table above is reached | The first interaction was the target |
| Example, switch | Above 0.07 | Moved by 0.02, the rollout ends at 0.045 |
| Example, block | Moved 0.01 | Moved 0.001 with the gripper within 0.1 |
| Code | `GoalTracker` | `SanityTracker`, using the authors' `check_state_difference` and the labels of `analyze_calvin_touch.py` |

Normal and `sanity_` behaviors can't be mixed in one config. `sanity_any` counts any interaction as a success, for the control runs.

Read the result from `Full_Distribution` in the results json. This is the number the paper plots, the share of each first interaction. It can differ slightly from `Success_Rate` when the robot moves several blocks at once.

**1. DynaGuide**, with `calvin_exp_configs_examples\sanity_switch_on.json`:
```bat
python dynaguide_scripts\run_dynaguide.py ^
    --output_folder results\sanity_switch_on ^
    --agent base_policy_300.pth ^
    --guidance dynamics_model_5900.pth ^
    --exp_setup_config calvin_exp_configs_examples\sanity_switch_on.json ^
    --horizon 400 ^
    --n_rollouts 50 ^
    --video_skip 10 ^
    --camera_names third_person ^
    --scale 1.5 ^
    --ss 4 ^
    --alpha 30 ^
    --no_live_render ^
    --no_collage
```

**2. Base policy without guidance**, on the same config. Use a new `--output_folder`, like `results\sanity_switch_on_base`, and `--scale 0 --ss 1` without `--alpha`.

**3. Control**, the base policy with a random start. Use a new `--output_folder`, `calvin_exp_configs_examples\sanity_control.json` and `--scale 0 --ss 1`.

**Settings from `paper_experiments/calvin_articulated_object.sh` and `calvin_movable_object.sh`.** All use `--ss 4 --horizon 400 --n_rollouts 50`:

| Task | `--scale` | `--alpha` |
|---|---|---|
| `switch_on`, `switch_off`, `door_left` | 1.5 | 30 |
| `button_on`, `button_off` | 1 | 30 |
| `drawer_open`, `drawer_close` | 1 | 40 |
| `door_right` | 2 | 15 |
| `red_lift`, `blue_lift`, `pink_lift` | 2 | 30 |

For another task, copy its config and write its `sanity_` name in `target_behaviors`. For `red_lift`, that is `sanity_red_displace`.

To score rollouts you already have by the paper's rules, pass a `sanity_` config to `evaluate_behaviors.py`.

**What still differs from the paper:**
- The paper averages 6 base policies with 50 rollouts each. We have one downloaded base policy, so expect more spread
- Only the `switch_on` guidance condition can be downloaded, the others have to be built from the CALVIN dataset
- The paper's images came from Linux with the EGL renderer. On Windows the camera uses the CPU TinyRenderer, whose images look different from the training images, so a lower score on Windows doesn't have to mean a bug

To check the guidance conditions, run:
```bat
python helper_scripts\show_guidance_images.py ^
    --dataset guidance_condition_switch_on_20.hdf5 ^
    --output results\guidance_images
```